# Fehlende Radinfra aus Mapillary-Fahrbahnmarkierungen → MapRoulette

Schlankere Fassung von `1_merge_mapillary-markings_osm-cycleways.ipynb`, im Aufbau
identisch zum Zwilling der Verkehrszeichen-Kampagne
([`../cycleway_complete_campaign/maproulette_tasks.ipynb`](../cycleway_complete_campaign/maproulette_tasks.ipynb)).
Die Logik liegt in [`../cw_campaign.py`](../cw_campaign.py), geteilt von beiden Kampagnen.
Tests: `uv run pytest test_cw_campaign.py` in `use_cases/`.

**Was gegenüber `1_` anders ist**

1. **Das verlinkte Foto ist das neueste.** `1_` nahm `images[-1]`, das letzte Element einer
   nicht nach Aufnahmezeit sortierten Liste. An 600 Stichproben war das in 84 % der Fälle
   nicht das neueste Bild, in 10 % lag über ein Jahr dazwischen.
2. **Der Challenge-Abzug wirkt jetzt wirklich.** In `1_` steht in Zelle 50
   `df_process_img = df_buffered_both_false.copy()` und die bereinigte Fassung
   `df_buffered_both_false_no_challenge` ist auskommentiert — der Abzug wurde also
   gerechnet und dann verworfen. Dadurch tauchten bereits als *Not an issue* oder
   *Already fixed* abgehakte Stellen erneut als Aufgaben auf.
3. **Eine Abstandsspalte statt zweier Pufferläufe.** `dist_cw_m` ersetzt
   `df_buffered_25`/`df_buffered_30`; die Aufgabentexte nennen dadurch den gemessenen
   Abstand statt der Schwelle.
4. **Sammelabfragen** bei Mapillary (50 Features pro Request) statt Einzelabrufen — an
   200 Features 1,9 s statt 30,5 s. Die `mapillary`-Library wird nicht mehr gebraucht.
5. **Vollständigkeits-Guard** beim Einlesen: fehlt ein Bundesland-Parquet, bricht der
   Lauf ab, statt still eine Teilmenge zu verarbeiten. Davor holt sich das Notebook den
   aktuellen Stand von data.vizsim.de, statt auf dem zu rechnen, was lokal herumliegt.
6. **`sidewalk:bicycle`** zählt als Radinfra — seit dem gemeinsamen
   [`../0_prepare_osm_network.ipynb`](../0_prepare_osm_network.ipynb) steht die Spalte
   auch hier zur Verfügung.

Der letzte Abschnitt stellt den vorherigen Stand der Aufgabendatei gegenüber.

## Einstellungen

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# cw_campaign.py liegt eine Ebene höher, weil beide Cycleway-Kampagnen es nutzen.
# Das ".." funktioniert, weil Jupyter und nbconvert das Arbeitsverzeichnis auf das
# Notebook-Verzeichnis setzen — dieselbe Annahme wie bei ../../output/ und ../utils/.
import sys

sys.path.insert(0, "..")
import cw_campaign as cw

# Muss zum Datum in ../0_prepare_osm_network.ipynb passen - das Notebook erzeugt
# die Radwege-Datei für beide Cycleway-Kampagnen.
set_date = "260915"

ordner_punkte = Path("../../output")
pfad_radwege = Path(f"../utils/processed_osm_files/processed_cycleways_germany_{set_date}.parquet")
pfad_grenze = Path("../utils/OSMB-germany.geojson.gz")
pfad_config = Path("../utils/config_mapillary_privat.json")

# Nur Markierungen, die nach diesem Datum zuletzt gesehen wurden ...
zuletzt_gesehen_nach = "2025-01-01"
# ... und die so viele Monate zwischen erster und letzter Sichtung liegen haben.
mindest_standzeit_monate = 9

# Fahrbahnmarkierungen liegen auch auf Wegen, die für Radverkehr nur freigegeben
# sind - deshalb zählt hier "yes" mit, anders als bei den Verkehrszeichen.
radinfra_werte = ("designated", "yes")

challenge_id = 53882

# Fester Name: MapRoulette zieht die Aufgaben immer aus demselben Pfad.
ziel_geojson = Path("maproulette_tasks_missing-cw_markings.geojson")

# Nur die Spalten lesen, die gebraucht werden - die map-feature-points-Parquets
# enthalten alle Feature-Klassen und sind entsprechend groß.
spalten = ["id", "value", "first_seen_at", "last_seen_at", "geometry"]

print("Markierungen:", ", ".join(cw.MARKIERUNGEN))
print("Prioritätsschwellen:", cw.PRIO_AB_DISTANZ)

## 1 · Markierungen holen und einlesen

Ohne den Sync-Schritt rechnet das Notebook auf dem, was zufällig lokal liegt. Beim Bau
dieses Notebooks am 20.09.2026 waren das Parquets vom **01.07.**, während der Server den
**17.09.** hatte — elf Wochen Unterschied, ohne jeden Hinweis.

In [ ]:
# Lädt nur, was lokal fehlt oder auf dem Server neuer ist.
metadata = cw.sync_features(ordner_punkte, prefix=cw.PREFIX_MARKIERUNGEN)
print("Datenstand:", metadata["processed_date"])

In [ ]:
# Sollwert aus dem Tile-Cache; ohne Tile-Cache (nur gespiegelte Parquets) None.
erwartet = cw.count_expected_states("../../prep/tile_cache/DE-*_tiles.json")
print("Tile-Cache kennt", erwartet, "Bundesländer")

marks = cw.load_features(
    ordner_punkte,
    prefix=cw.PREFIX_MARKIERUNGEN,
    values=cw.MARKIERUNGEN,
    columns=spalten,
    expect_files=erwartet,
)
marks.head()

In [ ]:
# Die Parquets sind nach Zoom-14-Kacheln geschnitten, Randkacheln ragen ins Ausland.
marks = cw.clip_to_boundary(marks, cw.load_boundary(pfad_grenze))

## 2 · Zeitliche Filter

In [ ]:
fig, achsen = plt.subplots(1, 2, figsize=(14, 3.5))
marks["last_seen_at"].str[:7].value_counts().sort_index().plot(kind="bar", ax=achsen[0], title="alle Markierungen")

marks = cw.filter_stable_signs(marks, zuletzt_gesehen_nach, mindest_standzeit_monate)

marks["last_seen_at"].str[:7].value_counts().sort_index().plot(
    kind="bar", ax=achsen[1],
    title=f"zuletzt gesehen nach {zuletzt_gesehen_nach}, ≥ {mindest_standzeit_monate} Monate",
)
plt.tight_layout()

## 3 · Abstand zur OSM-Radinfrastruktur

Statt zweier Pufferläufe (25 m / 30 m) eine Abstandsspalte. Gepuffert wird nur um die
paar tausend Punkte; die 5,8 Mio. OSM-Linien werden nicht umprojiziert, sondern nur die
Treffer des Vorfilters metrisch nachgemessen.

Einen Autobahn-Filter gibt es hier nicht — anders als bei den Verkehrszeichen war er in
`1_` durchgehend auskommentiert. Fahrbahnmarkierungen werden an Autobahnen praktisch
nicht erkannt.

In [ ]:
ways = gpd.read_parquet(pfad_radwege)
radinfra = cw.filter_cycle_infrastructure(ways, designated_werte=radinfra_werte)
print(f"Gesamtlänge: {cw.total_km(radinfra):,.2f} km".replace(",", "X").replace(".", ",").replace("X", "."))
del ways

In [ ]:
# Suchradius = größte Prioritätsschwelle; alles darüber interessiert nicht.
suchradius = max(schwelle for schwelle, _ in cw.PRIO_AB_DISTANZ)

marks["dist_cw_m"] = cw.distance_to_nearest(marks, radinfra, suchradius)
del radinfra

print(f"ohne Radinfra in {suchradius:.0f} m: {(marks.dist_cw_m == float('inf')).sum():>6}")

## 4 · Kandidaten und Priorität

MapRoulette-Priorität: 0 = High, 1 = Medium, 2 = Low.
Markierungen mit Radinfra näher als der kleinsten Schwelle fallen heraus.

In [ ]:
marks["prio"] = cw.assign_priority(marks["dist_cw_m"])

kandidaten = marks[marks["prio"].notna()].reset_index(drop=True)
kandidaten["prio_text"] = kandidaten["prio"].map(cw.PRIO_TEXT)
kandidaten["MapFeaturePoint"] = kandidaten["value"].map(cw.MARKIERUNGEN)

print(f"Kandidaten: {len(kandidaten)}")
kandidaten.groupby(["prio", "prio_text"]).size().to_frame("Anzahl")

In [ ]:
kandidaten.plot(column="prio", figsize=(6, 8), markersize=4, cmap="RdYlGn_r", legend=True)

## 5 · Bereits bestehende Aufgaben abziehen

In `1_` wurde dieser Schritt gerechnet und dann nicht verwendet — die bereinigte Fassung
war in der nächsten Zelle auskommentiert. Hier wirkt er.

In [ ]:
# Alles außer Fixed / Created / Skipped blockiert eine Neuanlage: was schon als
# "false positive" oder "already fixed" abgehakt wurde, soll nicht wiederkommen.
bestehende = cw.load_challenge_tasks(challenge_id, cw.load_token(pfad_config, "API_KEY_MAPROULETTE"))
print(f"blockierende Aufgaben in Challenge {challenge_id}: {len(bestehende)}")

kandidaten = cw.drop_near_existing_tasks(kandidaten, bestehende)
print(f"neu anzulegen: {len(kandidaten)}")

## 6 · Neuestes Mapillary-Bild

Derselbe Weg wie in der Verkehrszeichen-Kampagne: `fields=id,images{id,captured_at}` über
die Graph-API, bis zu 50 Features pro Anfrage. Map-Feature-Punkte und Verkehrszeichen
liegen bei Mapillary auf demselben Endpunkt, die Funktion ist also dieselbe.

In [ ]:
bilder = cw.newest_image_ids(kandidaten["id"], cw.load_token(pfad_config))

# Über den String-Index verbinden: Mapillary-IDs übersteigen 2**53 und würden
# als Zahl gerundet.
kandidaten = kandidaten.set_index(kandidaten["id"].astype(str)).join(bilder).reset_index(drop=True)

kandidaten[["id", "last_seen_at", "dist_cw_m", "image_id", "image_captured_at"]].head()

### Plausibilitätsprüfung

`last_seen_at` aus dem Parquet ist die letzte Aufnahme, auf der die Markierung erkannt
wurde. Wenn die Bildauswahl stimmt, muss das verlinkte Bild ungefähr von diesem Tag sein.

In [ ]:
abstand_tage = (
    kandidaten["image_captured_at"].dt.tz_localize(None) - pd.to_datetime(kandidaten["last_seen_at"])
).dt.days.abs()

print(abstand_tage.describe().to_string())
print(f"\nBild am selben Tag wie last_seen_at: {(abstand_tage <= 1).sum()} von {abstand_tage.notna().sum()}")

abstand_tage.plot(kind="hist", bins=40, figsize=(8, 3), title="Tage zwischen verlinktem Bild und last_seen_at")

## 7 · GeoJSON für MapRoulette schreiben

Die Datei behält ihren Namen, damit MapRoulette denselben Eingabepfad behalten kann.
Deshalb wird der bisherige Inhalt **vor** dem Überschreiben eingelesen.

In [ ]:
vorheriger_stand = cw.read_geojson(ziel_geojson)

aufgaben = cw.build_maproulette_geojson(kandidaten, cw.aufgabe_markierung)
cw.write_geojson(aufgaben, ziel_geojson)

print(aufgaben["features"][0]["properties"]["instruction"])

## 8 · Vergleich mit dem vorherigen Stand

Der interessante Wert ist **anderes Bild** — so viele Aufgaben verlinkten vorher ein
anderes (in aller Regel älteres) Foto. Stammt der vorherige Stand aus `1_`, ist ein
Mengenunterschied zu erwarten: dort wirkte der Challenge-Abzug nicht, und die Datei
wurde zu einem anderen Zeitpunkt erzeugt.

In [ ]:
if vorheriger_stand:
    cw.print_comparison(cw.compare_task_sets(vorheriger_stand, aufgaben))
else:
    print("kein vorheriger Stand vorhanden - Vergleich übersprungen")

---

## Challenge-Beschreibung (für MapRoulette)

## 🚲 Fehlende Radinfrastruktur anhand von Mapillary-Fahrbahnmarkierungen ergänzen (Deutschland)

Diese Challenge basiert auf automatisch erkannten Fahrrad-Symbolen auf der Fahrbahn aus
Mapillary-Bildern in Deutschland.

### 📌 Kriterien für jede Aufgabe

Nur Aufgaben, die **alle** folgenden Bedingungen erfüllen, wurden berücksichtigt:

- Das Fahrrad-Symbol wurde **in Mapillary erkannt**.
- Es wurde über **mindestens 9 Monate hinweg** gesehen.
- Die neueste Aufnahme stammt aus den letzten Monaten.
- Es existiert **kein OSM-"Radweg" innerhalb von 25 m** des Standortes.

### 🔍 Was du tun solltest

1. Öffne den Ort in **Mapillary** und **radinfra.de** sowie einem Editor.
2. Prüfe, ob an der Stelle eine **Radinfrastruktur fehlt**.
3. Falls ja, ergänze die passenden OSM-Tags — die Kopiervorlagen stehen in jeder Aufgabe.
4. Wenn bereits alles korrekt gemappt ist, kannst du die Aufgabe einfach **als erledigt markieren**.

🗺️ Vielen Dank für deine Hilfe beim Ausbau der Radinfrastruktur in OSM!

---

Task-Template für MapRoulette (Feld *Instruction*):

```
{{instruction}}
```